# Weather Impact Analysis

How weather conditions affect `arrival_delay`: rain, heavy rain, wind, snow and temperature.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.meteo as an

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("03_analysis_5-meteo")

%load_ext autoreload
%autoreload 2

## Wetterübersicht — alle Faktoren im Vergleich

Vergleich aller binären Wetterbedingungen auf einen Blick: Regen, starker Regen, Wind, Schnee — jeweils True vs. False. Zeigt welcher Faktor den grössten Einfluss hat.

In [ ]:
an.plot_weather_overview(lf_delay, cfg)

In [ ]:
show_df(an.table_weather_overview(lf_delay))

**Beobachtung:** Schnee hat den stärksten Einzeleffekt (+54.0s, OTP −10.9pp), gefolgt von Starkregen (+23.3s). Leichter Regen ist messbar aber moderat (+8.9s).

**`is_windy` — Feature-Idee, aber nicht nutzbar:**
Wind als Feature wurde untersucht, zeigt aber NaN in der gesamten Analyse. Ursache: Das Feature war in der Datenvorbereitung als Feature-Idee vorgesehen, wurde aber nie korrekt befüllt (vermutlich keine Tage mit Wind > 40km/h im Datensatz, oder das Feature wurde nie in die parquet-Dateien geschrieben).
Inhaltlich: Zürich ist durch Bebauung und Hügellagen relativ windgeschützt. Trams sind schwer und auf Schienen gebunden — Wind unter ~60 km/h hat kaum messbaren Betriebseffekt. **`is_windy` wird aus dem Feature-Set entfernt.** (→ F-WEAT-03)

**Wetter-Effekte im Überblick (3 valide Features):**
| Bedingung | Δ Delay (s) | OTP Normal | OTP Wetter | N |
|:---|---:|---:|---:|---:|
| Regen | +8.9 | 87.4% | 84.3% | 10.2M |
| Starkregen | +23.3 | 87.0% | 79.4% | 228k |
| **Schnee** | **+54.0** | **87.1%** | **76.1%** | 273k |

**Wichtige Einschränkung:** Das ist reine Korrelation, keine Kausalität. Alle Wetter-Features haben niedrige Korrelation mit `arrival_delay` (max 0.042). Wetter alleine erklärt wenig Varianz — Wetter-Features bleiben aber als schwache eigenständige Signale im Modell.

**Niederschlagsintensität** zeigt eine klare Dosis-Wirkungs-Beziehung: <2mm=62.6s → >10mm=89.5s. Das ist der stärkste und klarste Wettereffekt im Notebook.

→ Wetter-Flags behalten: `has_snow`, `precipitation`, `has_rain`, `has_heavy_rain`; `is_windy` entfernen; Multikollinearität mit Monat/Saison beachten.

## Temperatur — Kontinuierlicher Effekt

Temperatur in 5°C-Bins: zeigt ob der Effekt linear ist oder ob es Schwellwerte gibt (z.B. Frost unter 0°C).

In [ ]:
an.plot_temperature_precipitation(lf_delay, cfg)

In [ ]:
show_df(an.table_temperature_bins(lf_delay))

**Beobachtung:** Der Temperatureffekt ist monoton ansteigend — **kältere Temperaturen haben WENIGER Delay, wärmere MEHR**.

**Ø Delay nach Temperaturbereich (5°C-Bins):**
| Temperatur | Ø Delay (s) | OTP |
|:---|---:|---:|
| −5–0°C | 54.5 | 88.9% |
| **0–5°C** | **53.8** | **88.1%** (niedrigster Delay!) |
| 5–10°C | 55.4 | 87.4% |
| 15–20°C | 56.7 | 86.8% |
| 25–30°C | 59.7 | 85.3% |
| 35–40°C | 64.0 | 84.6% (n=15k — wenige Daten) |

**Kernbefund:** Die Kälte-Hypothese ist falsch — 0–5°C ist die beste Temperaturzone. Wärme verschlechtert die Pünktlichkeit graduell. Aber der Gesamteffekt ist klein: `is_hot` (>20°C) bringt nur **+2.0s Delta** (55.8s vs. 57.8s, OTP −1.1pp) — im Kontext aller Features ein schwaches Signal.

**Warum mehr Delay bei Wärme?**
- Sommer = mehr Freizeitverkehr, Tourismus, Events → vollere Trams, längere Boardingzeiten
- Gleisausdehnung bei Extremhitze (>30°C) → VBZ-Langsamfahrstellen (klassisches Problem)
- Im 35–40°C-Bin (n=15k) ist der Effekt am stärksten, aber die Datenbasis ist sehr dünn

**Kälte profitiert:** Konsistent mit F-TEMP-06 (Winter = beste Jahreszeit). Mögliche Ursache: weniger MIV bei Schnee/Frost kompensiert Halte-Verzögerungen.

→ `temperature` als kontinuierliches Feature; `is_hot` (>20°C) als binärer Flag; Effekt ist real aber klein (+2s) — nicht überbewerten.

## Feature: `is_hot`

Validierung des `is_hot`-Flags (temperature > 20°C) — binäre Vereinfachung des nicht-linearen Temperatureffekts für das Modell (F-WEAT-04).

In [ ]:
an.plot_is_hot(lf_delay, cfg)

In [ ]:
show_df(an.table_is_hot(lf_delay))

**Beobachtung:** Das `is_hot`-Feature (temperature > 20°C) validiert sich sauber.

**is_hot Vergleich:**
| Kategorie | Ø Delay (s) | OTP | N |
|:---|---:|---:|---:|
| Normal (≤20°C) | ~56s | ~87% | ~66M |
| Heiss (>20°C) | ~58s | ~86% | ~20M |

Der Effekt ist messbar aber moderat — `is_hot` ist ein nützlicher binärer Proxy für den kontinuierlichen Temperatureffekt. 
Die 20°C-Schwelle trennt zwei klar unterschiedliche Verteilungen, auch wenn der Effekt kleiner ist als der Schnee- oder Starkregen-Effekt.

## Daily Delay Timeline — Weather Events

Täglicher Delay-Verlauf pro Jahr — Schnee, Starkregen und Hitze als farbige Marker. Zeigt ob Delay-Spitzen mit Wetterereignissen zusammenfallen.

In [ ]:
an.plot_daily_delay_weather_timeline(lf_all, cfg)

In [ ]:
show_df(an.table_daily_delay_weather_timeline(lf_clean))

**Beobachtung:** Der tägliche Delay-Verlauf zeigt klare Wetter-Signaturen — aber Schnee-Tage ragen als Spitzen heraus, während Regen eher ein erhöhtes Grundrauschen erzeugt.

**Schnee-Spitzen** sind gut sichtbar als isolierte Peaks: einzelne Tage mit deutlich erhöhtem Delay, die mit blauen Markern zusammenfallen. Besonders ausgeprägt in den Wintermonaten Jan/Feb 2023 und 2024.

**Starkregen** erscheint als rote Häufung, oft im Herbst/Winter — weniger als singuläre Spitze, mehr als Phase erhöhter Delays.

**Temperatur** (gelbe Linie): Im Winter (niedrige Temperatur) sind die Delays tendenziell niedriger — konsistent mit F-WEAT-04. Sommerhitze (+20°C) korreliert mit leicht erhöhtem Grundniveau.

**Wichtige Einschränkung:** Wetter erklärt nicht alle Spitzen — Events (Berufsmesse, Stadtfest) können ähnliche Delay-Peaks erzeugen ohne Wetter-Marker. Die Kombination beider Notebooks ist nötig für eine vollständige Erklärung der Delay-Spitzen.

→ Schnee = scharfe, isolierte Peaks. Regen = diffuse Erhöhung. Wetter alleine erklärt die Varianz nicht vollständig (r < 0.05).

## Weather Impact Map — Stop Level

Δ Delay pro Haltestelle an Schnee- und Starkregen-Tagen vs. Normaltagen. Zeigt ob bestimmte Stadtteile oder Korridore besonders wetterempfindlich sind.

> **Warum Δ (Delta)?** Haltestellen haben sehr unterschiedliche Basis-Delays — eine Endhaltestelle hat strukturell mehr Delay als eine Innenstadthaltestelle. Δ = Wetter-Delay minus Normal-Delay macht Haltestellen vergleichbar: es zeigt den *zusätzlichen* Effekt des Wetters, unabhängig vom strukturellen Niveau.

In [ ]:
an.plot_weather_stop_map(lf_clean, flag="has_snow")
show_df(an.table_weather_stop_map(lf_clean, flag="has_snow"))

In [ ]:
an.plot_weather_stop_map(lf_clean, flag="has_heavy_rain")
show_df(an.table_weather_stop_map(lf_clean, flag="has_heavy_rain"))

**Beobachtung:** Die Karten zeigen zwei völlig unterschiedliche geografische Muster für Schnee und Regen.

**Schnee:** Konzentriert auf Kreis 4 (Hardbrücke/Hardplatz-Korridor) und Kreis 10 (Höngg/Wipkingen — erhöhte Lage, exponiert). Bahnhof Selnau als extremer Ausreisser (+190.9s, fast 4× Normal).

**Starkregen:** Konzentriert auf Kreis 5 (Escher Wyss / Toni-Areal Korridor, Limmat-Niederung). Toni-Areal +44s, Technopark +42s — industrielles Entwicklungsgebiet mit Drainage-Problemen.

→ Zwei verschiedene Vulnerabilitätskarten — dasselbe Netz, aber völlig andere Schwachstellen je nach Wetterereignis.

## Stadtkreis-Vergleich — Schnee vs. Starkregen

Δ Delay pro Stadtkreis für Schnee- vs. Starkregen-Tage nebeneinander. Zeigt welche Kreise besonders empfindlich auf welche Wetterbedingung reagieren.

In [ ]:
an.plot_district_weather_sensitivity(lf_clean, cfg)

In [ ]:
show_df(an.table_district_weather_sensitivity(lf_clean))

**Beobachtung:** Die Sortierung wechselt komplett zwischen Schnee und Regen — ein starkes Signal für topographische Ursachen.

| Stadtkreis | Δ Schnee | Δ Regen | Charakter |
|:---|---:|---:|:---|
| **Kreis 10** | **+89.4s** | +27.5s | Höngg — erhöhte Lage, exponiert |
| **Kreis 5** | +45.5s | **+36.8s** | Escher Wyss — Limmat-Niederung |
| Kreis 12 | +64.8s | +13.0s | Schwamendingen — suburban, erhöht |
| Kreis 4 | +69.9s | +23.5s | Aussersihl — urban, Hardbrücke-Korridor |

**Kernbefund:** Schnee trifft exponierte Höhenlagen (Kreis 10, 12, 4). Regen trifft Flusstäler und Niederlagen (Kreis 5, 9). Die Topographie bestimmt die Wetterempfindlichkeit.

## Haltestellen-Ranking — Schnee vs. Starkregen

Top 20 Haltestellen nach Δ Delay — sortiert, mit Durchschnittslinie. Kreise über dem Durchschnitt hervorgehoben. Beide Wetterereignisse separat — die Reihenfolge ändert sich zwischen Schnee und Starkregen.

In [ ]:
an.plot_stop_weather_ranking(lf_clean, cfg)

In [ ]:
show_df(an.table_stop_weather_ranking(lf_clean))

**Beobachtung:** Schnee und Regen treffen komplett verschiedene Haltestellen — kaum Überschneidung in den Top 20.

**Schnee-Ausreisser: Bahnhof Selnau (+190.9s)**
Normal 55.6s → Schnee 246.5s — fast 4× der Normalverzögerung. Einzige Haltestelle in dieser Größenordnung. Selnau liegt am Sihl-Ufer und ist Endpunkt mehrerer Linien; Verzögerungen akkumulieren sich dort. Starker Kandidat für operative Maßnahmen bei Schneeereignissen.

**Schnee-Cluster Kreis 4:** Helvetiaplatz, Bahnhof Hardbrücke, Hardplatz, Bäckeranlage — alle ~115s Delta. Dichter Korridor in Aussersihl.

**Schnee-Cluster Kreis 10:** 8 von 20 Top-Haltestellen in Höngg/Wipkingen — erhöhte Lage, Strecken besonders exponiert.

**Regen-Cluster Kreis 5:** 11 von 20 Top-Haltestellen im Escher Wyss / Toni-Areal Korridor. Toni-Areal +44.2s, Technopark +42.0s — Limmat-Niederung mit Drainage-Problemen bei Starkregen.

→ Räumlich trennscharf: Schnee = Höhenlagen, Regen = Limmat-Korridor.

## Linien-Betroffenheit — Welche Linien leiden am meisten?

Δ Delay pro Linie an Schnee- und Starkregen-Tagen. Zeigt welche Linien besonders wetterempfindlich sind — und ob die Reihenfolge zwischen Schnee und Regen wechselt.

In [ ]:
an.plot_line_weather_exposure(lf_clean, cfg)

In [ ]:
show_df(an.table_line_weather_exposure(lf_clean))

**Beobachtung:** Die Linien-Reihenfolge wechselt zwischen Schnee und Regen drastisch — das stärkste Muster im gesamten Wetter-Notebook.

| Linie | Δ Schnee | Δ Regen | Charakteristik |
|:---|---:|---:|:---|
| **13** | **+82.1s** | +37.7s | Hoch bei beiden — kreuzt beide Zonen |
| **9** | +75.9s | +10.0s | Schnee-Linie — Triemli, erhöhte Lagen |
| **17** | +7.7s | **+41.2s** | Regen-Linie — flache Limmat-Route durch Kreis 5 |
| **12** | +51.7s | +5.0s | Stärkster Gegensatz — Schwamendingen/Northeast |
| **4** | +44.2s | +40.4s | Ausgeglichen hoch — zentrale Achse |

**Kernbefund:** Linie 17 ist der klarste Beweis — Schnee fast irrelevant (+7.7s), Regen Platz 1 (+41.2s). Route führt flach durch den Escher Wyss / Kreis 5 Korridor. Linie 9 und 12 sind das Gegenteil: erhöhte Strecken, stark bei Schnee, kaum bei Regen.

→ Topographie des Linienverlaufs bestimmt die Wetterempfindlichkeit der Linie. Vorhersagemodell könnte davon profitieren: Linie + Wettertyp als Interaktionsterm.

## Multikollinearität — Wetter × Saison

In [ ]:
an.plot_multicollinearity_matrix(lf_delay, cfg)

In [ ]:
show_df(an.table_correlation_with_delay(lf_delay))

**Beobachtung:** Die Korrelationsmatrix bestätigt die erwarteten Zusammenhänge.

**Korrelation mit `arrival_delay` (abs. sortiert):**
- `has_snow` hat die stärkste Korrelation (~0.03–0.05) — absolut gering, aber konsistent
- Wetter-Features sind alle schwach korreliert mit Delay (r < 0.1) — Delay ist primär durch betriebliche Faktoren bestimmt
- `season` und `month` korrelieren erwartungsgemäss mit Wetter-Flags (Multikollinearität vorhanden)
- `has_rain` × `season`: negative Korrelation — Sommer (Season=3) ist trockener als Herbst

→ Wetter-Features sind schwache aber valide Prädiktoren; Multikollinearität mit Saison-Features beim Modellbau beachten.

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

`Präsentation`: **hot** = Kernbefund für Präsentation · **story** = gutes Narrativ · **—** = intern/Feature-Engineering

| ID | Finding | Präsentation |
|:---|:---|:---:|
| F-WEAT-01 | **Schnee** stärkster Wettereffekt: +54.0s, OTP 87.1%→76.1% — klarer Schwellwert-Effekt | **hot** |
| F-WEAT-02 | Starkregen: +23.3s, OTP −7.6pp. Niederschlagsintensität zeigt klare Dosis-Wirkungs-Beziehung: <2mm: 62.6s → >10mm: 89.5s | **story** |
| F-WEAT-03 | `is_windy` zeigt NaN — nie korrekt befüllt, inhaltlich kaum relevant. **Aus Feature-Set entfernt.** | — |
| F-WEAT-04 | Temperatureffekt monoton ansteigend. 0–5°C = bester Bereich (53.8s). `is_hot` (>20°C) = +2.0s — schwaches aber reales Signal | — |
| F-WEAT-05 | Alle Wetter-Features schwach korreliert (max r=0.042). Keine Multikollinearität mit Saison — unabhängige Signale | — |
| F-WEAT-06 | `precipitation` (r=0.036) und `has_snow` (r=0.038) nützlichste Features; `temperature` (r=0.018) schwächer | — |
| F-WEAT-07 | **Geografische Trennung:** Schnee trifft Höhenlagen (Kreis 10/4/12), Regen trifft Flusstäler (Kreis 5). Topographie bestimmt Vulnerabilität. Bahnhof Selnau extremer Schnee-Ausreisser: +190.9s (4× Normal) | **hot** |
| F-WEAT-08 | **Regen-Korridor Kreis 5:** 11 von top 20 Regen-Haltestellen im Escher Wyss / Toni-Areal / Limmat-Niederung. Toni-Areal +44.2s, Technopark +42.0s | **story** |
| F-WEAT-09 | **Linien reagieren komplett unterschiedlich:** Linie 17 Schnee +7.7s vs. Regen +41.2s (flache Limmat-Route). Linie 9 Schnee +75.9s vs. Regen +10.0s (erhöhte Lagen). Linie × Wettertyp als Interaktionsterm prüfen | **hot** |